In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, roc_curve)

plt.rcParams['axes.unicode_minus'] = False
df = pd.read_csv('processed_ml_dataset.csv', encoding='utf-8-sig')

feature_cols = ['min_temp','max_temp','avg_temp','avg_rhm','annual_rn',
                 'opt_temp_min','opt_temp_max','frost_limit_temp','opt_humidity',
                 'soil_ph_min','soil_ph_max']
rename_map = {
    '생육적온_최저(℃)':'opt_temp_min','생육적온_최고(℃)':'opt_temp_max',
    '한계생육온도(℃)':'frost_limit_temp','적정습도(%)':'opt_humidity',
    '토양pH_최저':'soil_ph_min','토양pH_최고':'soil_ph_max',
    '수익성(1-5)':'profit_score'
}
df = df.rename(columns=rename_map)

# 1. heatmap
numeric_cols = ['min_temp','max_temp','avg_temp','avg_rhm','annual_rn',
                 'opt_temp_min','opt_temp_max','frost_limit_temp','opt_humidity',
                 'soil_ph_min','soil_ph_max','profit_score','suitability']
plt.figure(figsize=(11,9))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, annot_kws={'size':8})
plt.title('Correlation Heatmap: Climate x Crop Growth Conditions')
plt.tight_layout(); plt.savefig('figs/01_heatmap.png', dpi=110); plt.close()

# 2. profit vs suitability
plt.figure(figsize=(7,5))
sns.barplot(data=df, x='profit_score', y='suitability', errorbar=None, color='#4C72B0')
plt.title('Profitability Score vs Suitability Rate')
plt.xlabel('Profitability Score (1-5)'); plt.ylabel('Suitability Rate')
plt.tight_layout(); plt.savefig('figs/02_profit_vs_suitability.png', dpi=110); plt.close()

# 3. min temp distribution across regions
plt.figure(figsize=(7,5))
sns.histplot(df.drop_duplicates('region')['min_temp'], bins=20, kde=True, color='#55A868')
plt.title('Regional Minimum Temperature Distribution (98 Korean Regions)')
plt.xlabel('Min Temperature (C)')
plt.tight_layout(); plt.savefig('figs/03_mintemp_dist.png', dpi=110); plt.close()

X = df[feature_cols].copy()
y = df['suitability']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train); X_test_s = scaler.transform(X_test)

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=200),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
}
results = []
roc_data = {}
for name, model in models.items():
    if name == 'LogisticRegression':
        model.fit(X_train_s, y_train); pred = model.predict(X_test_s); proba = model.predict_proba(X_test_s)[:,1]
    else:
        model.fit(X_train, y_train); pred = model.predict(X_test); proba = model.predict_proba(X_test)[:,1]
    results.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),
                     'Precision':precision_score(y_test,pred,zero_division=0),
                     'Recall':recall_score(y_test,pred,zero_division=0),
                     'F1':f1_score(y_test,pred,zero_division=0),
                     'ROC_AUC':roc_auc_score(y_test,proba)})
    fpr,tpr,_ = roc_curve(y_test, proba); roc_data[name]=(fpr,tpr,roc_auc_score(y_test,proba))
results_df = pd.DataFrame(results).sort_values('F1', ascending=False)

plt.figure(figsize=(7,6))
for name,(fpr,tpr,auc) in roc_data.items():
    plt.plot(fpr,tpr,label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--',alpha=0.3)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve Comparison')
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig('figs/04_roc_comparison.png', dpi=110); plt.close()

plt.figure(figsize=(7,5))
sns.barplot(data=results_df, x='Model', y='F1', color='#8172B2')
plt.title('F1-score by Model'); plt.xticks(rotation=15)
plt.tight_layout(); plt.savefig('figs/05_f1_comparison.png', dpi=110); plt.close()

param_grid = {'n_estimators':[100,200,300],'max_depth':[None,8,12],'min_samples_leaf':[1,3,5]}
grid = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)
best_rf = grid.best_estimator_
pred_tuned = best_rf.predict(X_test); proba_tuned = best_rf.predict_proba(X_test)[:,1]

importances = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(8,6))
sns.barplot(x=importances.values, y=importances.index, color='#64B5CD')
plt.title('Feature Importance (Tuned RandomForest)'); plt.xlabel('Importance')
plt.tight_layout(); plt.savefig('figs/06_feature_importance.png', dpi=110); plt.close()

cm = confusion_matrix(y_test, pred_tuned)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix (Tuned RandomForest)')
plt.tight_layout(); plt.savefig('figs/07_confusion_matrix.png', dpi=110); plt.close()

print("OK - figs regenerated with English labels")
print(results_df.to_string(index=False))
print("Best params:", grid.best_params_)
print("Tuned F1:", f1_score(y_test,pred_tuned), "Tuned ROC-AUC:", roc_auc_score(y_test,proba_tuned))

KeyboardInterrupt: 